# M5 HPO · Chronos-2 · State × Weight Ensemble · v1
## Full-data training, 12-way output selection

**Paradigm**: each of the **12 segment models** (4 weight tiers × 3 states) is fine-tuned on
**all 30 490 items** — preserving cross-item learning.  Forecast outputs are then
*selected* by state × weight tier for the ensemble.  This is the direct extension of
`ft_ensemble_v6` (which used 4 weight-tier segments on full data, best WRMSSE = 0.8364)
with an added geographic split.

### Why this design

| Design | Models | Train data / model | Selection |
|--------|--------|--------------------|-----------|
| ft_v6 | 4 | all 30 490 items | by weight tier |
| store_v1 | 10 | ~3 049 items / store | by store |
| **this** | **12** | **all 30 490 items** | **by state × weight** |
| 4×10 per-store-weight | 40 | ~3 049 items | by store × weight |

Keeps full cross-item learning, adds state-level config specialisation.

### Search-space priors (from v6 + sv1 studies)

- **CL**: {1,2,4,8,16,32} — CL≥64 never wins in either study
- **ft_steps**: {0,100,200,500,1000} — >1000 shows diminishing returns
- **ft_mode**: {lora, full} — lora dominates; full wins for med-low tier
- **ft_lr**: {1e-5,5e-5,1e-4,5e-4,1e-3,1e-2} — weight_high needs high lr
- **ft_bs**: {32,64,128,256}
- **is_weekend** (`is_friday`,`is_saturday`,`is_sunday`): **always** in `known_cov_cols`
- **event / price / snap**: per-segment 3-way choice — `none`, `past`, or `future`

### Cache correctness

Tag format `hpo_{zs|ft}_sw_full_...` unambiguously marks full-data runs.
A v6 fallback check reuses `hpo_{zs|ft}_weight_segs_...` when the covariate
combination matches (all selected covariates are future-known, no past covariates).
Store-level tags (`store_{id}_hpo_...`) are never read.


## 1 · Imports

In [1]:
import os, sys, gc, ctypes, contextlib, threading
sys.path.append("/home/nmwamsojo/tsfm-explo/src/jobs/")
from concurrent.futures import ThreadPoolExecutor, as_completed

import torch
import optuna
import pandas as pd
import numpy as np
from IPython.display import display

from m5_dataprep    import M5DataPipeline
from m5_exploration  import M5ExplorationSuite, DEFAULT_CHRONOS_CONFIG
from m5_evaluator    import M5Evaluator

print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)} ({props.total_memory//1024**3} GB)")


/home/nmwamsojo/tsfm-explo/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CUDA: True
  GPU 0: Quadro RTX 5000 (15 GB)
  GPU 1: Quadro RTX 5000 (15 GB)


In [2]:
for k in ["OMP","MKL","OPENBLAS","VECLIB_MAXIMUM","NUMEXPR"]:
    os.environ[f"{k}_NUM_THREADS"] = "4"
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

DEVICES = ([f"cuda:{i}" for i in range(torch.cuda.device_count())]
           if torch.cuda.is_available() else ["cpu"])
N_GPUS = len(DEVICES)
print(f"Active devices ({N_GPUS}): {DEVICES}")

if torch.cuda.is_available():
    for d in DEVICES: torch.zeros(1, device=d)
    torch.cuda.synchronize()
    print("CUDA contexts pre-initialised.")


Active devices (2): ['cuda:0', 'cuda:1']
CUDA contexts pre-initialised.


## 2 · Config

In [ ]:
DATA_PATH     = "/mnt/lab/datasets/M5/jointed_M5.parquet"
CALENDAR_PATH = "/mnt/lab/nmwamsojo/m5_data/calendar.csv"
ACTUALS_PATH  = "/mnt/lab/nmwamsojo/m5_data/sales_test_evaluation.csv"

DATA_TAG  = "sales_only"

# Two evaluation cutoffs — geometric mean of their WRMSSEs is the HPO objective.
# Both are strictly before 2016-05-22 so actuals come from the training parquet.
CUTOFF_1 = (pd.to_datetime("2016-05-22") - pd.Timedelta(days=28)).strftime("%Y-%m-%d")
CUTOFF_2 = (pd.to_datetime("2016-05-22") - pd.Timedelta(days=56)).strftime("%Y-%m-%d")
CUTOFFS  = [CUTOFF_1, CUTOFF_2]
print(f"Cutoff 1 : {CUTOFF_1}  (window {CUTOFF_1} + 28 d)")
print(f"Cutoff 2 : {CUTOFF_2}  (window {CUTOFF_2} + 28 d)")

# ── Search space — priors from v6 (segment-based) and sv1 (per-store) ────
CL_CHOICES       = [1, 2, 4, 8, 16, 32]
FT_STEPS_CHOICES = [0, 100, 200, 500, 1000]
FT_MODE_CHOICES  = ["lora", "full"]
FT_LR_CHOICES    = [1e-5, 5e-5, 1e-4, 5e-4, 1e-3, 1e-2]
FT_BATCH_CHOICES = [32, 64, 128, 256]

IS_WEEKEND_COLS = ["is_friday", "is_saturday", "is_sunday"]
EVENT_COLS      = ["event_name_1", "event_type_1"]
PRICE_COLS      = ["sell_price"]
SNAP_COLS       = ["snap_CA", "snap_TX", "snap_WI"]
COV_3WAY        = ["none", "past", "future"]

BATCH_SIZE = 64
N_TRIALS   = 300

OPTUNA_DB         = "/mnt/lab/nmwamsojo/optuna_state_weight_v1.db"
FORCE_FRESH_STUDY = False

WRAPPER = {
    "eval_metric":          "RMSSE",
    "enable_ensemble":      False,
    "skip_model_selection": True,
    "verbosity":            1,
}
CFG_BASE = {**DEFAULT_CHRONOS_CONFIG, "use_static": False}

print(f"\nCL choices  : {CL_CHOICES}")
print(f"ft_steps    : {FT_STEPS_CHOICES}")
print(f"Cov 3-way   : event / price / snap each in {COV_3WAY}")
print(f"Objective   : geometric mean of WRMSSE over {len(CUTOFFS)} cutoffs")
print(f"Budget      : {N_TRIALS} trials × 12 segments (96 Optuna params)")
print(f"DB          : {OPTUNA_DB}")

Cutoff 1 : 2016-04-24  (window 2016-04-24 + 28 d)
Cutoff 2 : 2016-03-27  (window 2016-03-27 + 28 d)

CL choices  : [1, 2, 4, 8, 16, 32]
ft_steps    : [0, 100, 200, 500, 1000]
Cov 3-way   : event / price / snap each in ['none', 'past', 'future']
Objective   : geometric mean of WRMSSE over 2 cutoffs
Budget      : 300 trials × 12 segments (96 Optuna params)
DB          : /mnt/lab/nmwamsojo/optuna_state_weight_v1.db


## 3 · Data

In [4]:
pipeline = M5DataPipeline(config={"tag": DATA_TAG})

# Load CUTOFF_1 data once — only to build the segment map.
# The large DataFrames (_hist_c1, _hist_trimmed_c1, _future_c1) are temp;
# they are released at the end of the seg_map cell below.
# static_df (item attributes, cutoff-independent) is kept permanently.
_hist_c1, _hist_trimmed_c1, _future_c1, static_df, _weights_c1 = (
    pipeline.get_prepared_data(DATA_PATH, CUTOFF_1, level=12, force_reprepare=False)
)
print(f"Startup load [{CUTOFF_1}]")
print(f"  hist_trimmed : {_hist_trimmed_c1.shape}")
print(f"  future       : {_future_c1.shape}")
print(f"  static_df    : {static_df.shape}  ← kept permanently (cutoff-independent)")
print(f"  pipeline     : kept for on-demand loads inside objective()")

--- Cache Hit: Data found in /mnt/lab/nmwamsojo/prepared_data/sales_only/level_12/20160424 ---
Startup load [2016-04-24]
  hist_trimmed : (45942500, 21)
  future       : (853720, 20)
  static_df    : (30490, 6)  ← kept permanently (cutoff-independent)
  pipeline     : kept for on-demand loads inside objective()


In [5]:
# Both cutoffs are strictly before 2016-05-22, so actuals for both forecast windows
# (CUTOFF+1 … CUTOFF+28) are present in the training parquet's "sold" column.
df_actual = (
    pd.read_parquet(DATA_PATH, columns=["id", "date", "sold"])
    .rename(columns={"sold": "sales_quantity"})
)
df_actual["id"] = (df_actual["id"].astype(str)
                   .str.replace("_evaluation", "", regex=False)
                   .str.replace("_validation",  "", regex=False))
print(f"Actuals: {df_actual.shape}  "
      f"{df_actual['date'].min().date()} → {df_actual['date'].max().date()}")

Actuals: (59181090, 3)  2011-01-29 → 2016-05-22


## 4 · Evaluator & Suite

In [6]:
AG_PATH_BASE = "/mnt/lab/nmwamsojo/autogluon_models/state_weight_v1"

# Suites only — lightweight path/config objects, no data stored inside them.
# M5Evaluator instances are built on-demand inside objective() per cutoff
# and deleted immediately after evaluation to avoid holding two full hist_df copies.
suites = {
    c: {
        d: M5ExplorationSuite(
            horizon  = 28,
            ag_path  = os.path.join(AG_PATH_BASE, c.replace("-", "")),
            base_dir = "/mnt/lab/nmwamsojo/prepared_data",
        )
        for d in DEVICES
    }
    for c in CUTOFFS
}
print(f"Suites     : {list(suites)}")
print("Evaluators : on-demand in objective() → released after each cutoff")

Suites     : ['2016-04-24', '2016-03-27']
Evaluators : on-demand in objective() → released after each cutoff


## 5 · State × Weight Segment Map

Weight tiers: equal-count quantiles of level-12 WRMSSE item weights.
State: extracted from item id (e.g. `FOODS_1_001_CA_1` → CA).

Items per tier ≈ 7 600; items per segment ≈ 1 900–3 500.
WRMSSE weight is heavily concentrated in the High tier (70.8% total),
especially CA-High (32.4%) — the single most impactful cell.


In [7]:
STATES       = ["CA", "TX", "WI"]
WEIGHT_TIERS = ["Low", "Medium-Low", "Medium-High", "High"]
TIER_KEY     = {t: t.lower().replace("-","_").replace(" ","_") for t in WEIGHT_TIERS}

# Build segment map from the temporary CUTOFF_1 weights
w12 = _weights_c1[_weights_c1["level"] == 12].copy()
w12["weight_tier"] = pd.qcut(w12["weight"], q=4, labels=WEIGHT_TIERS)
w12["state"] = w12["id"].str.extract(r"_(CA|TX|WI)_")[0]
w12 = w12.dropna(subset=["state"])

# Segment ID sets — key is (tier, state) matching SEGS ordering
SEG_IDS: dict[tuple, set] = {
    (t, s): set(w12[(w12["state"]==s) & (w12["weight_tier"]==t)]["id"].tolist())
    for t in WEIGHT_TIERS for s in STATES
}

# Pre-compute WRMSSE weight % per segment — used in results plots instead of w12
SEG_WRMSSE_PCT: dict[tuple, float] = {
    (t, s): w12[(w12["state"]==s) & (w12["weight_tier"]==t)]["weight"].sum() * 100
    for t in WEIGHT_TIERS for s in STATES
}

SEGS           = [(t, s) for t in WEIGHT_TIERS for s in STATES]
SEG_DEVICE_MAP = {seg: DEVICES[i % N_GPUS] for i, seg in enumerate(SEGS)}

print(f"{'Segment':>28}  {'Items':>6}  {'WRMSSE%':>8}  Device")
print("─" * 55)
for seg in SEGS:
    tier, state = seg
    print(f"  ({tier:>12}, {state})  {len(SEG_IDS[seg]):>6,}  "
          f"{SEG_WRMSSE_PCT[seg]:>7.2f}%  {SEG_DEVICE_MAP[seg]}")
print(f"  {'TOTAL':>16}  {sum(len(v) for v in SEG_IDS.values()):>6,}  100.00%")

# Release all startup DataFrames — only SEG_IDS, SEG_WRMSSE_PCT, static_df,
# df_actual, pipeline, and suites will remain in permanent memory.
del _hist_c1, _hist_trimmed_c1, _future_c1, _weights_c1, w12
gc.collect()
print("\nStartup DataFrames released.")

                     Segment   Items   WRMSSE%  Device
───────────────────────────────────────────────────────
  (         Low, CA)   2,581     0.70%  cuda:0
  (         Low, TX)   2,766     0.82%  cuda:1
  (         Low, WI)   2,279     0.67%  cuda:0
  (  Medium-Low, CA)   2,801     3.20%  cuda:1
  (  Medium-Low, TX)   2,479     2.75%  cuda:0
  (  Medium-Low, WI)   2,339     2.62%  cuda:1
  ( Medium-High, CA)   3,265     7.91%  cuda:0
  ( Medium-High, TX)   2,015     4.86%  cuda:1
  ( Medium-High, WI)   2,345     5.62%  cuda:0
  (        High, CA)   3,549    32.42%  cuda:1
  (        High, TX)   1,887    18.51%  cuda:0
  (        High, WI)   2,184    19.92%  cuda:1
             TOTAL  30,490  100.00%

Startup DataFrames released.


## 6 · HPO

### Cache design

Each model tag encodes only the **model config** (not the output segment),
because the same fully-trained model serves any segment that uses it.

```
hpo_zs_sw_full_cl{cl}_{cov_enc}
hpo_ft_sw_full_cl{cl}_steps{ft}__{mode}_lr{lr}_bs{bs}_{cov_enc}
```

`cov_enc` encodes is_weekend (always) plus the future/past choice for each group:
`iw[_eF|_eP][_pF|_pP][_sF|_sP]`  (e=event, p=price, s=snap; F=future, P=past).

**v6 fallback**: when all selected optional covariates are future-known (no past),
the equivalent v6 tag (`hpo_{zs/ft}_weight_segs_...`) is checked first.
If it exists, its forecasts (trained on full data) are reused directly.

Store-level tags (`store_*`) are **never** read.


In [8]:
# _model_load_lock: serialises ALL Chronos2.from_pretrained calls.
# accelerate.dispatch_model calls model.to(device) on meta tensors, which is not
# thread-safe — concurrent calls from different GPU threads both fail with
# "Cannot copy out of meta tensor". Serialising load+train is the safe fix.
# Cache hits (parquet reads) skip this lock and run fully in parallel.
_device_locks    = {d: threading.Lock() for d in DEVICES}
_model_load_lock = threading.Lock()

# ── Covariate helpers ─────────────────────────────────────────────────────

def _cov_cols(cov_event: str, cov_price: str, cov_snap: str) -> tuple[list, list]:
    known = list(IS_WEEKEND_COLS)
    past  = []
    for choice, cols in [(cov_event, EVENT_COLS),
                         (cov_price, PRICE_COLS),
                         (cov_snap,  SNAP_COLS)]:
        if choice == "future": known.extend(cols)
        elif choice == "past": past.extend(cols)
    return known, past

def _cov_enc(cov_event: str, cov_price: str, cov_snap: str) -> str:
    parts = ["iw"]
    if cov_event  == "future": parts.append("eF")
    elif cov_event == "past":  parts.append("eP")
    if cov_price  == "future": parts.append("pF")
    elif cov_price == "past":  parts.append("pP")
    if cov_snap   == "future": parts.append("sF")
    elif cov_snap  == "past":  parts.append("sP")
    return "_".join(parts)

# ── Tag helpers ───────────────────────────────────────────────────────────

def _make_tag(cl, ft_steps, ft_mode, ft_lr, ft_bs, cov_enc) -> str:
    if ft_steps == 0:
        return f"hpo_zs_sw_full_cl{cl}_{cov_enc}"
    lr_str = f"{ft_lr:.0e}".replace("-0", "-")
    return f"hpo_ft_sw_full_cl{cl}_steps{ft_steps}_{ft_mode}_lr{lr_str}_bs{ft_bs}_{cov_enc}"

def _v6_tag(cl, ft_steps, ft_mode, ft_lr, ft_bs,
            cov_event, cov_price, cov_snap) -> str | None:
    """v6 fallback — only valid when no past covariates (full-data namespace)."""
    if any(c == "past" for c in [cov_event, cov_price, cov_snap]):
        return None
    parts = ["is_weekend"]
    if cov_event == "future": parts.append("event")
    if cov_price == "future": parts.append("price")
    if cov_snap  == "future": parts.append("snap")
    v6_cov = "_".join(parts)
    if ft_steps == 0:
        return f"hpo_zs_weight_segs_cl{cl}_{v6_cov}"
    lr_str = f"{ft_lr:.0e}".replace("-0", "-")
    return f"hpo_ft_weight_segs_cl{cl}_steps{ft_steps}_{ft_mode}_lr{lr_str}_bs{ft_bs}_{v6_cov}"

def _forecast_path(tag: str, cutoff: str) -> str:
    return os.path.join(
        suites[CUTOFF_1][DEVICES[0]].base_dir, DATA_TAG, "level_12",
        cutoff.replace("-", ""), "models", tag, "forecasts.parquet",
    )

def _resolve_tag(cl, ft_steps, ft_mode, ft_lr, ft_bs,
                 cov_event, cov_price, cov_snap,
                 cutoff: str) -> tuple[str, bool]:
    """Priority: 1) sw_full  2) v6 weight_segs fallback.  Store tags never read."""
    enc     = _cov_enc(cov_event, cov_price, cov_snap)
    primary = _make_tag(cl, ft_steps, ft_mode, ft_lr, ft_bs, enc)
    if os.path.exists(_forecast_path(primary, cutoff)):
        return primary, True
    v6 = _v6_tag(cl, ft_steps, ft_mode, ft_lr, ft_bs, cov_event, cov_price, cov_snap)
    if v6 and os.path.exists(_forecast_path(v6, cutoff)):
        return v6, True
    return primary, False

def _deduplicate(seg_params: dict) -> tuple[dict, dict, dict]:
    """Collapse 12 segments to unique model configs; rebalance devices.

    Returns
    -------
    tag_params  : {tag: params_dict}   (unique models only)
    tag_device  : {tag: device}        (round-robin across GPUs after dedup)
    tag_to_segs : {tag: [seg, ...]}
    """
    tag_params:  dict[str, dict] = {}
    tag_to_segs: dict[str, list] = {}
    for seg in SEGS:
        p   = seg_params[seg]
        enc = _cov_enc(p["cov_event"], p["cov_price"], p["cov_snap"])
        tag = _make_tag(p["cl"], p["ft_steps"], p["ft_mode"], p["ft_lr"], p["ft_bs"], enc)
        p["_tag"] = tag
        if tag not in tag_params:
            tag_params[tag] = p
            tag_to_segs[tag] = [seg]
        else:
            tag_to_segs[tag].append(seg)
    tag_device = {tag: DEVICES[i % N_GPUS] for i, tag in enumerate(tag_params)}
    return tag_params, tag_device, tag_to_segs

print("Helpers ready.")

# ── Cache inventory ───────────────────────────────────────────────────────
for c in CUTOFFS:
    models_dir = os.path.join(
        suites[c][DEVICES[0]].base_dir, DATA_TAG, "level_12",
        c.replace("-", ""), "models"
    )
    if os.path.isdir(models_dir):
        tags    = os.listdir(models_dir)
        v6_hits = sum(1 for t in tags if "weight_segs" in t)
        sw_hits = sum(1 for t in tags if "sw_full" in t)
        print(f"  [{c}]  v6_weight_segs={v6_hits}  sw_full={sw_hits}")

Helpers ready.
  [2016-04-24]  v6_weight_segs=1  sw_full=2
  [2016-03-27]  v6_weight_segs=0  sw_full=1


In [9]:
def _run_one_model(
    tag:          str,
    is_hit:       bool,
    device:       str,
    cutoff:       str,
    hist_trimmed: "pd.DataFrame",
    future:       "pd.DataFrame",
    cl:           int,
    ft_steps:     int,
    ft_mode:      str,
    ft_lr:        float,
    ft_bs:        int,
    cov_event:    str,
    cov_price:    str,
    cov_snap:     str,
) -> "pd.DataFrame":
    """Train/load one model on FULL data for one cutoff.

    Lock discipline:
      cache hit  → no lock; both GPUs read parquet concurrently.
      cache miss → _model_load_lock (global); serialises from_pretrained across GPUs.
                   accelerate.dispatch_model is not thread-safe when called
                   concurrently from multiple threads, even on different devices.
    """
    device_idx = int(device.split(":")[-1]) if ":" in device else 0
    torch.cuda.set_device(device_idx)

    known_cols, past_cols = _cov_cols(cov_event, cov_price, cov_snap)
    cfg = {
        **CFG_BASE,
        "context_length":       cl,
        "fine_tune_steps":      ft_steps,
        "fine_tune_mode":       ft_mode  if ft_steps > 0 else "lora",
        "fine_tune_lr":         ft_lr    if ft_steps > 0 else 1e-4,
        "fine_tune_batch_size": ft_bs    if ft_steps > 0 else 128,
        "batch_size":           BATCH_SIZE,
        "known_cov_cols":       known_cols,
        "past_cov_cols":        past_cols,
        "device":               device,
    }

    c_suite = suites[cutoff][device]

    if is_hit:
        # Pure parquet read — no model loading, fully parallel.
        fcst = c_suite.run(
            hist_df=hist_trimmed, future_df=future, static_df=None,
            model="Chronos2", exp_config=cfg, exp_tag=tag,
            data_tag=DATA_TAG, cutoff_day=cutoff,
            wrapper_dict=WRAPPER, force_run=False,
        )
    else:
        # from_pretrained + fit: must not run concurrently across GPU threads.
        with _model_load_lock:
            c_suite._cached_tsdf.clear()
            c_suite._cached_future.clear()
            fcst = c_suite.run(
                hist_df=hist_trimmed, future_df=future, static_df=None,
                model="Chronos2", exp_config=cfg, exp_tag=tag,
                data_tag=DATA_TAG, cutoff_day=cutoff,
                wrapper_dict=WRAPPER, force_run=False,
            )

    with torch.cuda.device(device_idx):
        torch.cuda.empty_cache()
    return fcst


def _run_models_for_cutoff(
    tag_params:   dict,
    tag_device:   dict,
    cutoff:       str,
    hist_trimmed: "pd.DataFrame",
    future:       "pd.DataFrame",
) -> dict:
    """Run all unique models for ONE cutoff using the 2-GPU thread pool.

    hist_trimmed and future are the caller's DataFrames for this cutoff;
    they are NOT copied or stored — just referenced during the run.
    Returns {tag: full_forecast_df}; caller deletes this dict after evaluation.
    """
    jobs = []
    for tag, p in tag_params.items():
        device = tag_device[tag]
        _, is_hit = _resolve_tag(
            p["cl"], p["ft_steps"], p["ft_mode"], p["ft_lr"], p["ft_bs"],
            p["cov_event"], p["cov_price"], p["cov_snap"], cutoff,
        )
        jobs.append((tag, device, p, is_hit))

    n_hits   = sum(1 for *_, h in jobs if h)
    dev_dist = {d: sum(1 for j in jobs if j[1] == d) for d in DEVICES}
    print(f"    [{cutoff}]  {len(jobs)} models  "
          f"({n_hits} hits, {len(jobs)-n_hits} to run)  GPUs: {dev_dist}")

    results: dict[str, "pd.DataFrame"] = {}
    with ThreadPoolExecutor(max_workers=N_GPUS) as executor:
        fmap = {
            executor.submit(
                _run_one_model,
                tag, is_hit, device, cutoff, hist_trimmed, future,
                p["cl"], p["ft_steps"], p["ft_mode"], p["ft_lr"], p["ft_bs"],
                p["cov_event"], p["cov_price"], p["cov_snap"],
            ): tag
            for tag, device, p, is_hit in jobs
        }
        for fut in as_completed(fmap):
            results[fmap[fut]] = fut.result()
    return results


print("_run_one_model + _run_models_for_cutoff ready.")

_run_one_model + _run_models_for_cutoff ready.


In [10]:
def objective(trial: optuna.Trial) -> float:
    # ── 1. Sample params per segment ─────────────────────────────────────
    seg_params: dict[tuple, dict] = {}
    for seg in SEGS:
        tier, state = seg
        sk = f"sw_{state.lower()}_{TIER_KEY[tier]}"

        cl        = trial.suggest_categorical(f"CL_{sk}",        CL_CHOICES)
        ft_steps  = trial.suggest_categorical(f"ft_steps_{sk}",  FT_STEPS_CHOICES)
        ft_mode   = trial.suggest_categorical(f"ft_mode_{sk}",   FT_MODE_CHOICES)
        ft_lr     = trial.suggest_categorical(f"ft_lr_{sk}",     FT_LR_CHOICES)
        ft_bs     = trial.suggest_categorical(f"ft_bs_{sk}",     FT_BATCH_CHOICES)
        cov_event = trial.suggest_categorical(f"cov_event_{sk}", COV_3WAY)
        cov_price = trial.suggest_categorical(f"cov_price_{sk}", COV_3WAY)
        cov_snap  = trial.suggest_categorical(f"cov_snap_{sk}",  COV_3WAY)

        seg_params[seg] = {
            "cl": cl, "ft_steps": ft_steps, "ft_mode": ft_mode,
            "ft_lr": ft_lr, "ft_bs": ft_bs,
            "cov_event": cov_event, "cov_price": cov_price, "cov_snap": cov_snap,
        }

    # ── 2. Log trial summary ──────────────────────────────────────────────
    print(f"\n[Trial {trial.number}]")
    for tier in WEIGHT_TIERS:
        row = []
        for state in STATES:
            p = seg_params[(tier, state)]
            ft_str = (f"ft{p['ft_steps']}_{p['ft_mode']}_lr{p['ft_lr']:.0e}"
                      if p["ft_steps"] > 0 else "ZS")
            covs = f"e:{p['cov_event'][0]} p:{p['cov_price'][0]} s:{p['cov_snap'][0]}"
            row.append(f"{state} CL={p['cl']} {ft_str} [{covs}]")
        print(f"  {tier:>12}: " + "  |  ".join(row))

    # ── 3. Deduplicate once (model configs are cutoff-independent) ────────
    tag_params, tag_device, tag_to_segs = _deduplicate(seg_params)
    print(f"  {len(tag_params)} unique models (from 12 segments)")

    gc.collect(2)
    for d in DEVICES:
        with torch.cuda.device(d): torch.cuda.empty_cache()

    try:
        wrmsse_per_cutoff: dict[str, float] = {}

        # ── 4. Sequential cutoff loop — one cutoff's data in RAM at a time ─
        for cutoff in CUTOFFS:
            # Load this cutoff's data from parquet cache (~5 s, fast read)
            hist_df, hist_trimmed, future, _, wts = pipeline.get_prepared_data(
                DATA_PATH, cutoff, level=12, force_reprepare=False
            )
            evaluator = M5Evaluator(
                raw_train_df     = hist_df,
                trimmed_train_df = hist_trimmed,
                static_df        = static_df,
                weights_df       = wts,
                target_col       = "sales_quantity",
                price_col        = "sell_price",
            )

            # Run unique models for this cutoff (2-GPU parallel)
            model_fcsts = _run_models_for_cutoff(
                tag_params, tag_device, cutoff, hist_trimmed, future
            )

            # Assemble segment forecasts → ensemble
            seg_fcsts = []
            for seg in SEGS:
                tag  = seg_params[seg]["_tag"]
                fcst = model_fcsts[tag]
                seg_fcsts.append(fcst[fcst["id"].isin(SEG_IDS[seg])].copy())
            ensemble = pd.concat(seg_fcsts, ignore_index=True)
            seg_fcsts.clear()

            # Evaluate
            metrics = evaluator.evaluate_all(ensemble, df_actual)
            wrmsse_per_cutoff[cutoff] = float(metrics["WRMSSE"])

            # ── Release this cutoff's data immediately ────────────────────
            del hist_df, hist_trimmed, future, wts, evaluator
            del model_fcsts, ensemble, metrics
            gc.collect(2)
            for d in DEVICES:
                with torch.cuda.device(d): torch.cuda.empty_cache()

        # ── 5. Geometric mean — the HPO objective ─────────────────────────
        geo_mean = float(np.prod(list(wrmsse_per_cutoff.values())) ** (1.0 / len(CUTOFFS)))

        # ── 6. Store attrs ────────────────────────────────────────────────
        trial.set_user_attr("WRMSSE_geo", geo_mean)
        for c, v in wrmsse_per_cutoff.items():
            trial.set_user_attr(f"WRMSSE_{c.replace('-', '')}", v)
        for seg in SEGS:
            tier, state = seg
            sk = f"sw_{state.lower()}_{TIER_KEY[tier]}"
            for k, v in seg_params[seg].items():
                if not k.startswith("_"):
                    trial.set_user_attr(f"{k}_{sk}", v)

        scores_str = "  ".join(f"@{c}={v:.4f}" for c, v in wrmsse_per_cutoff.items())
        print(f"[Trial {trial.number}]  WRMSSE {scores_str}  →  geo={geo_mean:.4f}")
        return geo_mean

    except Exception as e:
        import traceback; traceback.print_exc()
        return float("inf")

    finally:
        gc.collect(2)
        for d in DEVICES:
            with torch.cuda.device(d): torch.cuda.empty_cache()
        try: ctypes.CDLL("libc.so.6").malloc_trim(0)
        except Exception: pass


print("objective() ready.")

objective() ready.


## 7 · Run Study

In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

if FORCE_FRESH_STUDY and os.path.exists(OPTUNA_DB):
    os.remove(OPTUNA_DB); print(f"Deleted: {OPTUNA_DB}")

study = optuna.create_study(
    direction  = "minimize",
    sampler    = optuna.samplers.TPESampler(
        seed=42, multivariate=True,
        n_startup_trials=50,
        constant_liar=True,
    ),
    study_name     = "chronos_state_weight_v1",
    storage        = f"sqlite:///{OPTUNA_DB}",
    load_if_exists = True,
)

from collections import Counter
sc = Counter(t.state.name for t in study.trials)
print("Study loaded — " + "  |  ".join(f"{v} {k.lower()}" for k, v in sc.items() if v))

_df = study.trials_dataframe(attrs=("number", "value", "params", "user_attrs"))
if not _df.empty and "value" in _df.columns:
    _df = _df[_df["value"].notna()].sort_values("value").reset_index(drop=True)
    _df.rename(columns={"value": "WRMSSE_geo"}, inplace=True)
    # Attach per-cutoff columns if stored
    c1_col = f"user_attrs_WRMSSE_{CUTOFF_1.replace('-','')}"
    c2_col = f"user_attrs_WRMSSE_{CUTOFF_2.replace('-','')}"
    show_cols = ["number", "WRMSSE_geo"]
    if c1_col in _df.columns: show_cols += [c1_col]
    if c2_col in _df.columns: show_cols += [c2_col]
    if not _df.empty:
        display(
            _df[show_cols].head(20).style
            .format({c: "{:.4f}" for c in show_cols if c != "number"})
            .background_gradient(subset=["WRMSSE_geo"], cmap="RdYlGn_r")
            .set_caption(f"Top-20 by geo-mean WRMSSE ({len(_df)} stored trials)")
        )
        print(f"Best geo-mean = {_df['WRMSSE_geo'].min():.4f}  "
              f"(trial #{int(_df.iloc[0]['number'])})")
    else:
        print("No completed trials yet.")
else:
    print("No completed trials yet.")
print("─" * 70)

study.optimize(objective, n_trials=N_TRIALS, n_jobs=1, gc_after_trial=True)
try:
    print(f"\nBest WRMSSE (geo-mean) : {study.best_value:.4f}")
except ValueError:
    print("\nNo completed trials after optimization.")

Deleted: /mnt/lab/nmwamsojo/optuna_state_weight_v1.db
Study loaded — 
No completed trials yet.
──────────────────────────────────────────────────────────────────────

[Trial 0]
           Low: CA CL=2 ft100_lora_lr1e-03 [e:f p:f s:p]  |  TX CL=4 ZS [e:n p:f s:p]  |  WI CL=32 ft500_full_lr5e-04 [e:p p:f s:p]
    Medium-Low: CA CL=32 ft200_lora_lr5e-04 [e:f p:n s:f]  |  TX CL=2 ZS [e:f p:n s:n]  |  WI CL=32 ft1000_full_lr1e-02 [e:n p:n s:f]
   Medium-High: CA CL=1 ZS [e:f p:p s:n]  |  TX CL=4 ZS [e:n p:f s:f]  |  WI CL=32 ft200_lora_lr1e-03 [e:p p:f s:f]
          High: CA CL=16 ft200_lora_lr1e-03 [e:f p:n s:f]  |  TX CL=16 ft100_lora_lr5e-04 [e:f p:n s:f]  |  WI CL=1 ZS [e:p p:n s:f]
  12 unique models (from 12 segments)
--- Cache Hit: Data found in /mnt/lab/nmwamsojo/prepared_data/sales_only/level_12/20160424 ---
  [CPU] CuPy not available — scale computation on 18 CPU cores.
  Building hierarchy scales and weights …
    [2016-04-24]  12 models  (2 hits, 10 to run)  GPUs: {'cuda:0': 6,

fit known_cov_cols : ['is_friday', 'is_saturday', 'is_sunday', 'sell_price']
Fitting model with hyperparameters: 

	Chronos2: {'model_path': 'autogluon/chronos-2', 'context_length': 32, 'device': 'cuda:0', 'batch_size': 64, 'fine_tune_steps': 500, 'fine_tune_mode': 'full', 'fine_tune_lr': 0.0005, 'fine_tune_batch_size': 256}
[DBG] ag_train items=30490  shape=(45942500, 10)
[DBG] ag_future items=30490  shape=(853720, 4)
[hpo_ft_sw_full_cl32_steps500_full_lr5e-4_bs256_iw_eP_pF_sP] Forecast saved → /mnt/lab/nmwamsojo/prepared_data/sales_only/level_12/20160424/models/hpo_ft_sw_full_cl32_steps500_full_lr5e-4_bs256_iw_eP_pF_sP/forecasts.parquet
[hpo_ft_sw_full_cl32_steps200_lora_lr5e-4_bs256_iw_eF_sF] Forecasts not found at /mnt/lab/nmwamsojo/prepared_data/sales_only/level_12/20160424/models/hpo_ft_sw_full_cl32_steps200_lora_lr5e-4_bs256_iw_eF_sF/forecasts.parquet…
[hpo_ft_sw_full_cl32_steps200_lora_lr5e-4_bs256_iw_eF_sF] Running Chronos2 experiment…
  known covariates : ['is_friday', 'is_sa

fit known_cov_cols : ['is_friday', 'is_saturday', 'is_sunday', 'event_name_1', 'event_type_1', 'snap_CA', 'snap_TX', 'snap_WI']
Fitting model with hyperparameters: 

	Chronos2: {'model_path': 'autogluon/chronos-2', 'context_length': 32, 'device': 'cuda:1', 'batch_size': 64, 'fine_tune_steps': 200, 'fine_tune_mode': 'lora', 'fine_tune_lr': 0.0005, 'fine_tune_batch_size': 256}
[DBG] ag_train items=30490  shape=(45942500, 9)
[DBG] ag_future items=30490  shape=(853720, 8)
[hpo_ft_sw_full_cl32_steps200_lora_lr5e-4_bs256_iw_eF_sF] Forecast saved → /mnt/lab/nmwamsojo/prepared_data/sales_only/level_12/20160424/models/hpo_ft_sw_full_cl32_steps200_lora_lr5e-4_bs256_iw_eF_sF/forecasts.parquet
[hpo_zs_sw_full_cl2_iw_eF] Forecasts not found at /mnt/lab/nmwamsojo/prepared_data/sales_only/level_12/20160424/models/hpo_zs_sw_full_cl2_iw_eF/forecasts.parquet…
[hpo_zs_sw_full_cl2_iw_eF] Running Chronos2 experiment…
  known covariates : ['is_friday', 'is_saturday', 'is_sunday', 'event_name_1', 'event_type

fit known_cov_cols : ['is_friday', 'is_saturday', 'is_sunday', 'event_name_1', 'event_type_1']
Fitting model with hyperparameters: 

	Chronos2: {'model_path': 'autogluon/chronos-2', 'context_length': 2, 'device': 'cuda:0', 'batch_size': 64}
[DBG] ag_train items=30490  shape=(45942500, 6)
[DBG] ag_future items=30490  shape=(853720, 5)
[hpo_zs_sw_full_cl2_iw_eF] Forecast saved → /mnt/lab/nmwamsojo/prepared_data/sales_only/level_12/20160424/models/hpo_zs_sw_full_cl2_iw_eF/forecasts.parquet
[hpo_ft_sw_full_cl32_steps1000_full_lr1e-2_bs256_iw_sF] Forecasts not found at /mnt/lab/nmwamsojo/prepared_data/sales_only/level_12/20160424/models/hpo_ft_sw_full_cl32_steps1000_full_lr1e-2_bs256_iw_sF/forecasts.parquet…
[hpo_ft_sw_full_cl32_steps1000_full_lr1e-2_bs256_iw_sF] Running Chronos2 experiment…
  known covariates : ['is_friday', 'is_saturday', 'is_sunday', 'snap_CA', 'snap_TX', 'snap_WI']


fit known_cov_cols : ['is_friday', 'is_saturday', 'is_sunday', 'snap_CA', 'snap_TX', 'snap_WI']
Fitting model with hyperparameters: 

	Chronos2: {'model_path': 'autogluon/chronos-2', 'context_length': 32, 'device': 'cuda:1', 'batch_size': 64, 'fine_tune_steps': 1000, 'fine_tune_mode': 'full', 'fine_tune_lr': 0.01, 'fine_tune_batch_size': 256}
[DBG] ag_train items=30490  shape=(45942500, 7)
[DBG] ag_future items=30490  shape=(853720, 6)
[hpo_ft_sw_full_cl32_steps1000_full_lr1e-2_bs256_iw_sF] Forecast saved → /mnt/lab/nmwamsojo/prepared_data/sales_only/level_12/20160424/models/hpo_ft_sw_full_cl32_steps1000_full_lr1e-2_bs256_iw_sF/forecasts.parquet
[hpo_zs_sw_full_cl1_iw_eF_pP] Forecasts not found at /mnt/lab/nmwamsojo/prepared_data/sales_only/level_12/20160424/models/hpo_zs_sw_full_cl1_iw_eF_pP/forecasts.parquet…
[hpo_zs_sw_full_cl1_iw_eF_pP] Running Chronos2 experiment…
  known covariates : ['is_friday', 'is_saturday', 'is_sunday', 'event_name_1', 'event_type_1']
  past  covariates : ['

...

fit known_cov_cols : ['is_friday', 'is_saturday', 'is_sunday', 'sell_price', 'snap_CA', 'snap_TX', 'snap_WI']
Fitting model with hyperparameters: 

	Chronos2: {'model_path': 'autogluon/chronos-2', 'context_length': 4, 'device': 'cuda:1', 'batch_size': 64}
[DBG] ag_train items=30490  shape=(45942500, 8)
[DBG] ag_future items=30490  shape=(853720, 7)
[hpo_zs_sw_full_cl4_iw_pF_sF] Forecast saved → /mnt/lab/nmwamsojo/prepared_data/sales_only/level_12/20160424/models/hpo_zs_sw_full_cl4_iw_pF_sF/forecasts.parquet
[hpo_ft_sw_full_cl32_steps200_lora_lr1e-3_bs64_iw_eP_pF_sF] Forecasts not found at /mnt/lab/nmwamsojo/prepared_data/sales_only/level_12/20160424/models/hpo_ft_sw_full_cl32_steps200_lora_lr1e-3_bs64_iw_eP_pF_sF/forecasts.parquet…
[hpo_ft_sw_full_cl32_steps200_lora_lr1e-3_bs64_iw_eP_pF_sF] Running Chronos2 experiment…
  known covariates : ['is_friday', 'is_saturday', 'is_sunday', 'sell_price', 'snap_CA', 'snap_TX', 'snap_WI']
  past  covariates : ['event_name_1', 'event_type_1']


fit known_cov_cols : ['is_friday', 'is_saturday', 'is_sunday', 'sell_price', 'snap_CA', 'snap_TX', 'snap_WI']
Fitting model with hyperparameters: 

	Chronos2: {'model_path': 'autogluon/chronos-2', 'context_length': 32, 'device': 'cuda:0', 'batch_size': 64, 'fine_tune_steps': 200, 'fine_tune_mode': 'lora', 'fine_tune_lr': 0.001, 'fine_tune_batch_size': 64}
[DBG] ag_train items=30490  shape=(45942500, 10)
[DBG] ag_future items=30490  shape=(853720, 7)
[hpo_ft_sw_full_cl32_steps200_lora_lr1e-3_bs64_iw_eP_pF_sF] Forecast saved → /mnt/lab/nmwamsojo/prepared_data/sales_only/level_12/20160424/models/hpo_ft_sw_full_cl32_steps200_lora_lr1e-3_bs64_iw_eP_pF_sF/forecasts.parquet
[hpo_ft_sw_full_cl16_steps200_lora_lr1e-3_bs64_iw_eF_sF] Forecasts not found at /mnt/lab/nmwamsojo/prepared_data/sales_only/level_12/20160424/models/hpo_ft_sw_full_cl16_steps200_lora_lr1e-3_bs64_iw_eF_sF/forecasts.parquet…
[hpo_ft_sw_full_cl16_steps200_lora_lr1e-3_bs64_iw_eF_sF] Running Chronos2 experiment…
  known covaria

fit known_cov_cols : ['is_friday', 'is_saturday', 'is_sunday', 'event_name_1', 'event_type_1', 'snap_CA', 'snap_TX', 'snap_WI']
Fitting model with hyperparameters: 

	Chronos2: {'model_path': 'autogluon/chronos-2', 'context_length': 16, 'device': 'cuda:1', 'batch_size': 64, 'fine_tune_steps': 200, 'fine_tune_mode': 'lora', 'fine_tune_lr': 0.001, 'fine_tune_batch_size': 64}
[DBG] ag_train items=30490  shape=(45942500, 9)
[DBG] ag_future items=30490  shape=(853720, 8)
[hpo_ft_sw_full_cl16_steps200_lora_lr1e-3_bs64_iw_eF_sF] Forecast saved → /mnt/lab/nmwamsojo/prepared_data/sales_only/level_12/20160424/models/hpo_ft_sw_full_cl16_steps200_lora_lr1e-3_bs64_iw_eF_sF/forecasts.parquet
[hpo_ft_sw_full_cl16_steps100_lora_lr5e-4_bs64_iw_eF_sF] Forecasts not found at /mnt/lab/nmwamsojo/prepared_data/sales_only/level_12/20160424/models/hpo_ft_sw_full_cl16_steps100_lora_lr5e-4_bs64_iw_eF_sF/forecasts.parquet…
[hpo_ft_sw_full_cl16_steps100_lora_lr5e-4_bs64_iw_eF_sF] Running Chronos2 experiment…
  kn

fit known_cov_cols : ['is_friday', 'is_saturday', 'is_sunday', 'event_name_1', 'event_type_1', 'snap_CA', 'snap_TX', 'snap_WI']
Fitting model with hyperparameters: 

	Chronos2: {'model_path': 'autogluon/chronos-2', 'context_length': 16, 'device': 'cuda:0', 'batch_size': 64, 'fine_tune_steps': 100, 'fine_tune_mode': 'lora', 'fine_tune_lr': 0.0005, 'fine_tune_batch_size': 64}
[DBG] ag_train items=30490  shape=(45942500, 9)
[DBG] ag_future items=30490  shape=(853720, 8)
[hpo_ft_sw_full_cl16_steps100_lora_lr5e-4_bs64_iw_eF_sF] Forecast saved → /mnt/lab/nmwamsojo/prepared_data/sales_only/level_12/20160424/models/hpo_ft_sw_full_cl16_steps100_lora_lr5e-4_bs64_iw_eF_sF/forecasts.parquet
[hpo_zs_sw_full_cl1_iw_eP_sF] Forecasts not found at /mnt/lab/nmwamsojo/prepared_data/sales_only/level_12/20160424/models/hpo_zs_sw_full_cl1_iw_eP_sF/forecasts.parquet…
[hpo_zs_sw_full_cl1_iw_eP_sF] Running Chronos2 experiment…
  known covariates : ['is_friday', 'is_saturday', 'is_sunday', 'snap_CA', 'snap_TX',

fit known_cov_cols : ['is_friday', 'is_saturday', 'is_sunday', 'sell_price']
Fitting model with hyperparameters: 

	Chronos2: {'model_path': 'autogluon/chronos-2', 'context_length': 4, 'device': 'cuda:1', 'batch_size': 64}
[DBG] ag_train items=30490  shape=(45088780, 8)
[DBG] ag_future items=30490  shape=(1707440, 4)
[hpo_zs_sw_full_cl4_iw_pF_sP] Forecast saved → /mnt/lab/nmwamsojo/prepared_data/sales_only/level_12/20160327/models/hpo_zs_sw_full_cl4_iw_pF_sP/forecasts.parquet
[hpo_ft_sw_full_cl32_steps500_full_lr5e-4_bs256_iw_eP_pF_sP] Forecasts not found at /mnt/lab/nmwamsojo/prepared_data/sales_only/level_12/20160327/models/hpo_ft_sw_full_cl32_steps500_full_lr5e-4_bs256_iw_eP_pF_sP/forecasts.parquet…
[hpo_ft_sw_full_cl32_steps500_full_lr5e-4_bs256_iw_eP_pF_sP] Running Chronos2 experiment…
  known covariates : ['is_friday', 'is_saturday', 'is_sunday', 'sell_price']
  past  covariates : ['event_name_1', 'event_type_1', 'snap_CA', 'snap_TX', 'snap_WI']
fit known_cov_cols : ['is_friday',

## 8 · Results

In [ ]:
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid", font_scale=1.0)

df_t = study.trials_dataframe(attrs=("number", "value", "params", "user_attrs"))
if not df_t.empty and "value" in df_t.columns:
    df_t = df_t[df_t["value"].notna()].sort_values("value").reset_index(drop=True)
    df_t.rename(columns={"value": "WRMSSE_geo"}, inplace=True)
    c1_col = f"user_attrs_WRMSSE_{CUTOFF_1.replace('-','')}"
    c2_col = f"user_attrs_WRMSSE_{CUTOFF_2.replace('-','')}"

n_done = len(df_t) if not df_t.empty else 0
if n_done:
    best_geo = df_t["WRMSSE_geo"].min()
    best_c1  = df_t[c1_col].min() if c1_col in df_t.columns else float("nan")
    best_c2  = df_t[c2_col].min() if c2_col in df_t.columns else float("nan")
    print(f"Completed : {n_done}")
    print(f"Best geo  : {best_geo:.4f}")
    print(f"  @{CUTOFF_1} : {best_c1:.4f}")
    print(f"  @{CUTOFF_2} : {best_c2:.4f}")
else:
    print("No trials yet.")

In [ ]:
if df_t.empty or n_done < 5:
    print("Not enough trials to plot."); raise SystemExit

fig, axes = plt.subplots(2, 3, figsize=(17, 10))

# ── (a) Convergence ───────────────────────────────────────────────────────
ax = axes[0, 0]
dt = df_t.sort_values("number")
ax.plot(dt["number"], dt["WRMSSE_geo"].cummin(), "#2171b5", lw=2, label="geo-mean (obj)")
if c1_col in dt.columns:
    ax.plot(dt["number"], dt[c1_col].cummin(), "#74c476", lw=1.2, ls="--", label=f"@{CUTOFF_1}")
if c2_col in dt.columns:
    ax.plot(dt["number"], dt[c2_col].cummin(), "#fd8d3c", lw=1.2, ls="--", label=f"@{CUTOFF_2}")
ax.axhline(0.8364, color="grey", ls=":", lw=1.2, label="v6 best (0.8364)")
ax.set(xlabel="Trial", ylabel="Best WRMSSE", title="(a) Convergence")
ax.legend(fontsize=8); ax.grid(True, alpha=0.4)

# ── (b) CL heatmap ────────────────────────────────────────────────────────
ax = axes[0, 1]
cl_vals = {}
for tier in WEIGHT_TIERS:
    for state in STATES:
        sk  = f"sw_{state.lower()}_{TIER_KEY[tier]}"
        col = f"params_CL_{sk}"
        if col in df_t.columns:
            cl_vals[(tier, state)] = df_t.groupby(col)["WRMSSE_geo"].mean().idxmin()
if cl_vals:
    hm = (pd.DataFrame({"tier":[k[0] for k in cl_vals], "state":[k[1] for k in cl_vals],
                         "best_CL":[v for v in cl_vals.values()]})
          .pivot(index="tier", columns="state", values="best_CL").reindex(WEIGHT_TIERS))
    sns.heatmap(hm, ax=ax, cmap="Blues", annot=True, fmt="g", linewidths=0.4,
                cbar_kws={"label": "Best CL (mean geo WRMSSE)"})
    ax.set_title("(b) Best-mean CL per segment")

# ── (c) ft_steps heatmap ──────────────────────────────────────────────────
ax = axes[0, 2]
ft_vals = {}
for tier in WEIGHT_TIERS:
    for state in STATES:
        sk  = f"sw_{state.lower()}_{TIER_KEY[tier]}"
        col = f"params_ft_steps_{sk}"
        if col in df_t.columns:
            ft_vals[(tier, state)] = df_t.groupby(col)["WRMSSE_geo"].mean().idxmin()
if ft_vals:
    hm2 = (pd.DataFrame({"tier":[k[0] for k in ft_vals], "state":[k[1] for k in ft_vals],
                          "best_ft":[v for v in ft_vals.values()]})
           .pivot(index="tier", columns="state", values="best_ft").reindex(WEIGHT_TIERS))
    sns.heatmap(hm2, ax=ax, cmap="Purples", annot=True, fmt="g", linewidths=0.4,
                cbar_kws={"label": "Best ft_steps"})
    ax.set_title("(c) Best-mean ft_steps per segment")

# ── (d) Covariate frequency ───────────────────────────────────────────────
ax = axes[1, 0]
top50 = df_t.head(50)
cov_freq: dict[str, int] = {}
for seg in SEGS:
    tier, state = seg
    sk = f"sw_{state.lower()}_{TIER_KEY[tier]}"
    for grp in ["event", "price", "snap"]:
        col = f"params_cov_{grp}_{sk}"
        if col in top50.columns:
            for v in top50[col].dropna():
                key = f"{grp}_{v}"; cov_freq[key] = cov_freq.get(key, 0) + 1
cs = pd.Series(cov_freq).sort_values(ascending=False)
colors_bar = ["#2171b5" if "future" in k else "#74c476" if "past" in k else "#bdbdbd" for k in cs.index]
cs.plot(kind="bar", ax=ax, color=colors_bar, alpha=0.85)
ax.set_title("(d) Cov choice frequency — top-50 trials")
ax.tick_params(axis="x", rotation=40); ax.grid(True, axis="y", alpha=0.4)

# ── (e) WRMSSE weight heatmap — uses pre-computed SEG_WRMSSE_PCT ──────────
ax = axes[1, 1]
hm3 = (pd.DataFrame({"tier":[k[0] for k in SEG_WRMSSE_PCT],
                      "state":[k[1] for k in SEG_WRMSSE_PCT],
                      "wt_pct":[v for v in SEG_WRMSSE_PCT.values()]})
       .pivot(index="tier", columns="state", values="wt_pct").reindex(WEIGHT_TIERS))
sns.heatmap(hm3, ax=ax, cmap="Oranges", annot=True, fmt=".1f", linewidths=0.4,
            cbar_kws={"label": "% WRMSSE weight"})
ax.set_title("(e) WRMSSE weight per segment (%)")

# ── (f) Per-cutoff scatter ────────────────────────────────────────────────
ax = axes[1, 2]
if c1_col in df_t.columns and c2_col in df_t.columns:
    ax.scatter(df_t[c1_col], df_t[c2_col],
               c=df_t["WRMSSE_geo"], cmap="RdYlGn_r", alpha=0.6, s=20)
    ax.axvline(0.8364, color="grey", ls=":", lw=1.0)
    ax.axhline(0.8364, color="grey", ls=":", lw=1.0)
    ax.set(xlabel=f"WRMSSE @{CUTOFF_1}", ylabel=f"WRMSSE @{CUTOFF_2}",
           title="(f) Per-cutoff scatter (colour = geo-mean)")
    ax.grid(True, alpha=0.4)
else:
    ax.hist(df_t["WRMSSE_geo"].clip(upper=2.0), bins=30, color="#2171b5", alpha=0.7)
    ax.axvline(df_t["WRMSSE_geo"].min(), color="red", ls="-", lw=1.5)
    ax.set(xlabel="WRMSSE geo-mean", title="(f) Geo-mean histogram")

fig.suptitle(f"State × Weight v1 — {n_done} trials  |  geo-mean WRMSSE over {len(CUTOFFS)} cutoffs",
             fontsize=12, fontweight="bold")
fig.tight_layout(); plt.show()

In [ ]:
if not df_t.empty:
    try:
        best_t = study.best_trial
    except ValueError:
        print("No completed trials yet.")
    else:
        print(f"Best trial #{best_t.number}  WRMSSE={best_t.value:.4f}")
        print()
        print(f"  {'Segment':>22}  {'CL':>4}  {'ft':>6}  {'mode':>5}  "
              f"{'lr':>8}  {'bs':>4}  event  price  snap")
        print("─"*80)
        for tier in WEIGHT_TIERS:
            for state in STATES:
                sk = f"sw_{state.lower()}_{TIER_KEY[tier]}"
                cl = best_t.params.get(f"CL_{sk}")
                ft = best_t.params.get(f"ft_steps_{sk}")
                md = best_t.params.get(f"ft_mode_{sk}") if ft else "—"
                lr = best_t.params.get(f"ft_lr_{sk}")   if ft else "—"
                bs = best_t.params.get(f"ft_bs_{sk}")   if ft else "—"
                ev = best_t.params.get(f"cov_event_{sk}")
                pr = best_t.params.get(f"cov_price_{sk}")
                sn = best_t.params.get(f"cov_snap_{sk}")
                print(f"  ({tier:>12}, {state})  {str(cl):>4}  {str(ft):>6}  "
                      f"{str(md):>5}  {str(lr):>8}  {str(bs):>4}  "
                      f"{str(ev):<6} {str(pr):<6} {str(sn):<6}")

In [ ]:
from pathlib import Path
if not df_t.empty:
    out = Path(f"/mnt/lab/nmwamsojo/results/"
               f"m5_hpo_state_weight_v1_{CUTOFF_1.replace('-','')}.csv")
    out.parent.mkdir(parents=True, exist_ok=True)
    df_t.to_csv(out, index=False)
    print(f"Saved → {out}  ({len(df_t)} rows)")